In [36]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [37]:
word_index=imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

In [38]:
model=load_model('SIMPLE_RNN_imdb (1).h5')

In [39]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 500, 128)       │       128,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 161,027 (629.02 KB)

 Trainable params: 161,025 (629.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [40]:
# Step 2: Helper Functions

# Function to decode reviews
max_features=1000
def decode_review(encoded_review):
    return ' '.join(
        reverse_word_index.get(i - 3, '?')
        for i in encoded_review
    )


# Function to preprocess user input
def preprocess_text(text):
    words = text.lower().split()

    encoded_review = []

    for word in words:
        word_id = word_index.get(word)

        if word_id is None or word_id + 3 >= max_features:
            encoded_review.append(2)   # Unknown word
        else:
            encoded_review.append(word_id + 3)

    # Make length = 500
    padded_review = sequence.pad_sequences(
        [encoded_review],
        maxlen=500
    )

    return padded_review

In [41]:

# Prediction function

def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)

    prediction = model.predict(preprocessed_input)

    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'

    return sentiment, prediction[0][0]

In [42]:
review = "This movie was fantastic! the acting was great and the plot was thrilling"

result = predict_sentiment(review)

print(f'Review : {review}')
print(f'Result : {result}')
## print(f'Prediction Score : {score}')


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 537ms/step
Review : This movie was fantastic! the acting was great and the plot was thrilling
Result : ('Positive', np.float32(0.92794))


In [43]:
review = "This movie was over acted ! the acting was bad and the plot was copied from hollywood"

result = predict_sentiment(review)

print(f'Review : {review}')
print(f'Result : {result}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Review : This movie was over acted ! the acting was bad and the plot was copied from hollywood
Result : ('Negative', np.float32(0.37330723))
